# H27 Surrogate/Mixture Diversity Comparison

Compare already-generated H27 samples from `CNF`, `HTBAL_CNF_MIXPRIOR`, and `HTBAL_CNF_GUIDED` using the same simulator validation and the same c_l1-free 62k scalable dynamic-reference assignment.

This notebook does not retrain models. It answers: does the surrogate/mixture-prior branch design improve dynamic corridor coverage relative to the baseline CNF under the corrected c_l1-free evaluator?

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('not in Colab or Drive already unavailable:', repr(exc))

from pathlib import Path
import os

PROJECT_ROOT_CANDIDATES = [
    Path.cwd(),
    Path('/content/drive/MyDrive/Colab Notebooks/final_submission_h27_flow_main_20260623'),
    Path('/content/drive/MyDrive/final_submission_h27_flow_main_20260623'),
]
PROJECT_ROOT = next((p for p in PROJECT_ROOT_CANDIDATES if (p / 'scripts').exists()), Path.cwd())
os.chdir(PROJECT_ROOT)
print('PROJECT_ROOT =', PROJECT_ROOT)


In [ ]:
# Existing full run to compare. This should already contain generated NPZs.
OUT_ROOT = Path('outputs/experiments/20260622_h27_c_l1_free_surrogate_cnf')
RUN_NAME = 'full'
RUN_DIR = OUT_ROOT / RUN_NAME
BASE_CONDITION = 'CFAST_ORANGE3'
METHODS = ['CNF', 'HTBAL_CNF_MIXPRIOR', 'HTBAL_CNF_GUIDED']
CONDITION_SETS = [f'{BASE_CONDITION}_{m}' for m in METHODS]

REFERENCE_ASSIGNMENTS = Path('outputs/scalable_mode_clustering_20260604/csv/dynamic_family_assignments.csv')
REFERENCE_FEATURE_CACHE_NPZ = Path('outputs/scalable_mode_clustering_20260604/npz/scalable_dynamic_reference_features_t101_eta_path_pop.npz')
REFERENCE_FEATURE_CACHE_MANIFEST = Path('outputs/scalable_mode_clustering_20260604/npz/scalable_dynamic_reference_features_t101_eta_path_pop_manifest.json')
REFERENCE_FEATURE_CACHE = REFERENCE_FEATURE_CACHE_NPZ if REFERENCE_FEATURE_CACHE_NPZ.exists() else REFERENCE_FEATURE_CACHE_MANIFEST

# Validation knobs. Keep FORCE_* false to reuse successful prior work.
RUN_SIMULATOR_VALIDATION = True
RUN_SCALABLE_REFERENCE_ASSIGNMENT = True
FORCE_VALIDATION = False
FORCE_ASSIGNMENT = False
VALIDATION_TARGETS = 'ALL'
VALIDATION_N_PER_TARGET = -1
VALIDATION_SELECTION = 'head'
VALIDATION_LAMBDA_REORG = 35.0
VALIDATION_T_MAX = 50.0
VALIDATION_DT = 0.5
SAVE_TRAJECTORIES = True
ASSIGN_TARGET_MATCH_ONLY = True
ASSIGN_SUCCESS_ONLY = True
SEED = 20260622
THREAD_ENV = {
    'OMP_NUM_THREADS': '2',
    'OPENBLAS_NUM_THREADS': '2',
    'MKL_NUM_THREADS': '2',
    'NUMEXPR_NUM_THREADS': '2',
}

print('run dir:', RUN_DIR)
print('condition sets:', CONDITION_SETS)
print('reference assignments:', REFERENCE_ASSIGNMENTS, REFERENCE_ASSIGNMENTS.exists())
print('reference feature cache:', REFERENCE_FEATURE_CACHE, REFERENCE_FEATURE_CACHE.exists())

In [ ]:
def run(cmd, env=None):
    cmd = list(map(str, cmd))
    print('\n$ ' + ' '.join(cmd), flush=True)
    merged_env = os.environ.copy()
    merged_env.update(THREAD_ENV)
    if env:
        merged_env.update(env)
    merged_env['PYTHONUNBUFFERED'] = '1'
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=merged_env)
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
    returncode = process.wait()
    if returncode != 0:
        raise RuntimeError(f'command failed with exit code {returncode}: {cmd}')
    return subprocess.CompletedProcess(cmd, returncode)

def find_script(name):
    candidates = [Path('scripts') / name, Path.cwd() / 'scripts' / name]
    for p in candidates:
        if p.exists():
            return p
    for root in [Path.cwd(), Path('/content/drive/MyDrive')]:
        if not root.exists():
            continue
        try:
            for p in root.rglob(name):
                if p.parent.name in ('scripts', 'script'):
                    return p
        except Exception as exc:
            print('script search skipped:', root, repr(exc))
    raise FileNotFoundError(name)

VALIDATE_SCRIPT = find_script('validate_h27_cfast_generated_simulator_latest.py') if (Path('scripts') / 'validate_h27_cfast_generated_simulator_latest.py').exists() else find_script('validate_h27_cfast_generated_simulator.py')
ASSIGN_SCRIPT = find_script('assign_h27_generated_to_scalable_dynamic_reference.py')
print('validate:', VALIDATE_SCRIPT)
print('assign:', ASSIGN_SCRIPT)
run(['python3', '-m', 'py_compile', VALIDATE_SCRIPT])
run(['python3', '-m', 'py_compile', ASSIGN_SCRIPT])

try:
    import qutip  # noqa: F401
    print('qutip ok')
except ModuleNotFoundError:
    print('installing qutip...')
    run([sys.executable, '-m', 'pip', 'install', '-q', 'qutip'])

In [ ]:
assert RUN_DIR.exists(), f'missing run dir: {RUN_DIR}'
assert REFERENCE_ASSIGNMENTS.exists(), f'missing reference assignments: {REFERENCE_ASSIGNMENTS}'
assert REFERENCE_FEATURE_CACHE.exists(), f'missing reference feature cache: {REFERENCE_FEATURE_CACHE}'

for condition_set in CONDITION_SETS:
    generated = RUN_DIR / f'{condition_set}_generated_samples.npz'
    print(condition_set, 'generated exists=', generated.exists(), generated)
    assert generated.exists(), f'missing generated NPZ: {generated}'

In [ ]:
for condition_set in CONDITION_SETS:
    generated = RUN_DIR / f'{condition_set}_generated_samples.npz'
    val_out = RUN_DIR / f'simulator_validation_{condition_set}'
    detail = val_out / 'csv' / f'{generated.stem}_simulator_validation_detail.csv'
    validation_summary = val_out / 'csv' / f'{generated.stem}_simulator_validation_summary.csv'
    trajectories = val_out / 'npz' / f'{generated.stem}_sampled_trajectories.npz'

    if RUN_SIMULATOR_VALIDATION:
        if validation_summary.exists() and detail.exists() and trajectories.exists() and not FORCE_VALIDATION:
            print('reuse validation:', condition_set, validation_summary)
        else:
            cmd = [
                'python3', '-u', VALIDATE_SCRIPT,
                '--generated', generated,
                '--out-dir', val_out,
                '--condition-set', condition_set,
                '--targets', VALIDATION_TARGETS,
                '--n-per-target', VALIDATION_N_PER_TARGET,
                '--selection', VALIDATION_SELECTION,
                '--lambda-reorg', VALIDATION_LAMBDA_REORG,
                '--t-max', VALIDATION_T_MAX,
                '--dt', VALIDATION_DT,
                '--print-every', 20,
            ]
            if SAVE_TRAJECTORIES:
                cmd.append('--save-trajectories')
            run(cmd)

    if RUN_SCALABLE_REFERENCE_ASSIGNMENT:
        assert detail.exists(), f'missing validation detail for assignment: {detail}'
        assert trajectories.exists(), f'missing trajectories for assignment: {trajectories}'
        assign_out = RUN_DIR / f'scalable_reference_assignment_{condition_set}'
        existing = sorted((assign_out / 'csv').glob('*_scalable_reference_summary.csv')) if (assign_out / 'csv').exists() else []
        if existing and not FORCE_ASSIGNMENT:
            print('reuse assignment:', condition_set, existing[0])
        else:
            cmd = [
                'python3', '-u', ASSIGN_SCRIPT,
                '--detail', detail,
                '--trajectories', trajectories,
                '--condition-set', condition_set,
                '--run-label', f'{OUT_ROOT.name}/{RUN_NAME}',
                '--reference-assignments', REFERENCE_ASSIGNMENTS,
                '--reference-feature-cache', REFERENCE_FEATURE_CACHE,
                '--out-dir', assign_out,
                '--components', 'eta,path,pop',
                '--generated-chunk-size', 128,
                '--seed', SEED,
            ]
            if ASSIGN_SUCCESS_ONLY:
                cmd.append('--success-only')
            if ASSIGN_TARGET_MATCH_ONLY:
                cmd.append('--target-match-only')
            run(cmd)
print('validation and c_l1-free assignment stage complete')

In [ ]:
import pandas as pd
from IPython.display import display

rows = []
validation_rows = []
for method, condition_set in zip(METHODS, CONDITION_SETS):
    generated = RUN_DIR / f'{condition_set}_generated_samples.npz'
    val_summary = RUN_DIR / f'simulator_validation_{condition_set}' / 'csv' / f'{generated.stem}_simulator_validation_summary.csv'
    assign_dir = RUN_DIR / f'scalable_reference_assignment_{condition_set}' / 'csv'
    summary_paths = sorted(assign_dir.glob('*_scalable_reference_summary.csv'))
    assert summary_paths, f'missing scalable reference summary for {condition_set}: {assign_dir}'
    df = pd.read_csv(summary_paths[0])
    df.insert(0, 'method', method)
    df.insert(1, 'condition_set_short', condition_set)
    rows.append(df)
    if val_summary.exists():
        vdf = pd.read_csv(val_summary)
        vdf.insert(0, 'method', method)
        vdf.insert(1, 'condition_set_short', condition_set)
        validation_rows.append(vdf)

cmp = pd.concat(rows, ignore_index=True)
val_cmp = pd.concat(validation_rows, ignore_index=True) if validation_rows else pd.DataFrame()
out_csv = RUN_DIR / 'c_l1_free_surrogate_vs_baseline_diversity_comparison.csv'
cmp.to_csv(out_csv, index=False)
print('wrote:', out_csv)

cols = [
    'method', 'target', 'n_assigned', 'dynamic_family_count', 'reference_dynamic_family_count',
    'dynamic_family_coverage_fraction', 'top_dynamic_family_id', 'top_dynamic_family_fraction',
    'dynamic_family_entropy_norm', 'structural_mode_count', 'structural_mode_coverage_fraction',
    'nearest_reference_distance_median', 'nearest_reference_distance_p90',
]
display(cmp[cols].sort_values(['target', 'method']))

metric_cols = ['dynamic_family_coverage_fraction', 'top_dynamic_family_fraction', 'dynamic_family_entropy_norm', 'nearest_reference_distance_median']
base = cmp[cmp['method'].eq('CNF')][['target'] + metric_cols].rename(columns={c: f'baseline_{c}' for c in metric_cols})
delta = cmp.merge(base, on='target', how='left')
for c in metric_cols:
    delta[f'delta_vs_CNF_{c}'] = delta[c] - delta[f'baseline_{c}']
delta_cols = ['method', 'target'] + metric_cols + [f'delta_vs_CNF_{c}' for c in metric_cols]
display(delta[delta_cols].sort_values(['target', 'method']))

In [ ]:
lines = []
lines.append('# H27 c_l1-free Surrogate vs Baseline Diversity Comparison')
lines.append('')
lines.append('Evaluator: simulator validation followed by nearest-reference assignment to the 62k c_l1-free dynamic reference (`eta_t,path_t,pop_t[:, :7]`).')
lines.append('Lower `top_dynamic_family_fraction` and higher `dynamic_family_entropy_norm` indicate less dynamic-family collapse. Higher coverage is useful, but should be read with the reference family count.')
lines.append('')
for target in sorted(cmp['target'].unique()):
    sub = cmp[cmp['target'].eq(target)].copy()
    if sub.empty:
        continue
    selected_entropy = sub.sort_values('dynamic_family_entropy_norm', ascending=False).iloc[0]
    selected_top = sub.sort_values('top_dynamic_family_fraction', ascending=True).iloc[0]
    baseline = sub[sub['method'].eq('CNF')]
    lines.append(f'## {target}')
    if not baseline.empty:
        b = baseline.iloc[0]
        lines.append(f"- baseline CNF: n={int(b.n_assigned)}, families={int(b.dynamic_family_count)}/{int(b.reference_dynamic_family_count)}, top={b.top_dynamic_family_id} {b.top_dynamic_family_fraction:.3f}, entropy={b.dynamic_family_entropy_norm:.3f}")
    lines.append(f"- highest measured entropy in this run: {selected_entropy.method} entropy={selected_entropy.dynamic_family_entropy_norm:.3f}, top_fraction={selected_entropy.top_dynamic_family_fraction:.3f}, coverage={selected_entropy.dynamic_family_coverage_fraction:.3f}")
    lines.append(f"- lowest top-family collapse: {selected_top.method} top_fraction={selected_top.top_dynamic_family_fraction:.3f}, entropy={selected_top.dynamic_family_entropy_norm:.3f}, coverage={selected_top.dynamic_family_coverage_fraction:.3f}")
    lines.append('')

report = RUN_DIR / 'reports' / 'c_l1_free_surrogate_vs_baseline_diversity_comparison_kr.md'
report.parent.mkdir(parents=True, exist_ok=True)
report.write_text('\n'.join(lines), encoding='utf-8')
print('wrote:', report)
print('\n'.join(lines[:80]))